# Tutorial 1 – Setup & first look at your corpus
*MIR & ML for Audio · TU Berlin · WiSe 2026/27*

**Notebook basics:** `Shift+Enter` runs a cell. The **kernel** (your `mirml` environment) keeps all variables in memory; cells can run in any order but might depend on each other -> before handing anything in: **Restart Kernel & Run All**! 

| Block | Topic 
|---|---
| 0 | Check your environment
| 1 | Project structure & paths
| 2 | Files & duplicates
| 3 | Load & inspect the chart table
| 4 | Missing & odd values: count, share, per year
| 5 | Plotting with matplotlib

---
## 0 · Check your environment
All imports in **one cell at the top** to check availability in current env

`sys.executable` shows which Python runs – the path must contain `mirml`. If not: *Select Kernel* (top right) → `mirml`.

In [ ]:
import sys
import shutil                      # copy / delete files
from pathlib import Path           # OS-independent paths

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(sys.executable)
print("pandas", pd.__version__)

---
## 1 · Project structure & paths
Open the **course folder** *mirml* (or whatever you named it) in your editor (*File → Open Folder…*):
```
mirml/                              <- course folder
├── T01_setup_and_corpus.ipynb      <- place your scripts/ notebooks here
├── data/
│   └── <your-genre>/
│       ├── <genre>.csv             <- chart table (Top 1000 per year)
│       ├── previews/               <- New folder for task 3: one MP3 per track ID
│       └── 2013/top100/<beatport_track_id>.mp3 … 2023/   <- as downloaded, can be deleted after task 3
└── output/                         <- figures & tables you create
```
- **Never rename or edit** the downloaded files. New files go into new folders (`previews/`, `output/`).
- **Relative paths** (`data/house`) instead of absolute ones (`C:\Users\…`) → runs on any computer.
- **`pathlib.Path`** works on Windows *and* macOS: join with `/`.

In [ ]:
# ── CONFIG: the only cell you need to edit ───────────────────────────────
from pathlib import Path

CORPUS_DIR: Path = Path("data") / "your-corpus-name"   # <- name of your corpus folder
# ─────────────────────────────────────────────────────────────────────────
PREVIEW_DIR: Path = CORPUS_DIR / "previews"
OUTPUT_DIR: Path = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

assert CORPUS_DIR.is_dir(), f"Not found: {CORPUS_DIR.resolve()} → check CORPUS_DIR"
CHART_CSV = next(CORPUS_DIR.glob("*.csv"))   # the one CSV in your corpus folder
print(CHART_CSV)

---
## 2 · Files & duplicates
A track that charts in several years lies in several year folders → same file name (= `beatport_track_id`), same audio. Goal: **one copy per track** in `previews/`.

| Tool | What it does |
|---|---|
| `folder.glob("*/top100/*.mp3")` | all MP3s in the year folders (`*` = any name) |
| `p.name`, `p.stem`, `p.parent.parent.name` | `"123.mp3"`, `"123"`, year folder |
| `duplicated()`, `drop_duplicates()`, `nunique()` | find / remove / count repeated values (pandas) |
| `folder.mkdir(exist_ok=True)` | create a folder (no error if it exists) |
| `shutil.copy2(src, dst)` | copy a file (keeps its timestamp) |
| `dst.exists()` | skip files already copied → cell can be rerun safely |

⚠ `rglob("*.mp3")` would search **all** subfolders – including `previews/` once it exists. Use a pattern that matches only what you mean.

In [ ]:
from pathlib import Path

year_files: list[Path] = sorted(CORPUS_DIR.glob(pattern="*/top100/*.mp3"))   # sorted -> same order on every OS
print(len(year_files), "files in the year folders")

p: Path = year_files[0] # index for first file
print(p.name, "|", p.stem, "|", p.parent.parent.name)

In [ ]:
# Duplicates – toy example
from typing import Any

ids: Any = pd.Series(["111", "222", "111", "333", "222"])
print(ids.duplicated().tolist())        # True from the 2nd occurrence on
print(ids.drop_duplicates().tolist())   # first occurrence of each value
print(ids.nunique(), "unique values")

In [ ]:
# Copy ONE file – skip if already there
PREVIEW_DIR.mkdir(exist_ok=True)
dst = PREVIEW_DIR / p.name
if not dst.exists():
    shutil.copy2(p, dst)
print(dst, dst.exists())

---
## 3 · Load & inspect the chart table
**pandas** `DataFrame` = table: rows (chart entries) × columns (variables), each column with one **dtype**.
First look, always: `.shape` → size, `.head()` → content, `.info()` → dtypes and **non-null counts**.

In [ ]:
from typing import Any

charts: Any = pd.read_csv(CHART_CSV)
print(charts.shape)   # (rows, columns)
charts.head() # head() always prints first 5 rows for inspection, head(10) prints the first 10 respectively

In [ ]:
charts.info() # prints Columns, Non-Null Count and Data type

---
## 4 · Missing & odd values: count, share, per year
**Think first – there is more than one valid approach:**
- **What counts as missing?** `NaN` – but also placeholders (`0`, `""`, `"unknown"`). Look before you count.
- **Absolute vs. relative:** counts (*how many*) vs. shares (*what fraction*). Shares make groups of different size comparable.
- **Which reference (denominator)?** Metadata: all chart entries. Audio: only the **Top 100** – there are no previews for ranks 101–1000.
- **Which unit?** Chart entries or unique tracks? Per field, per year, field × year?

The tools below run on a **toy table** – transfer them to `charts`.

In [ ]:
toy = pd.DataFrame({
    "year":     [2013, 2013, 2013, 2014, 2014],
    "rank":     [1, 2, 150, 1, 2],
    "track_id": [11, 22, 33, 44, 55],
    "artist":   ["A", None, "C", "D", ""],
    "bpm":      [128, 0, np.nan, 124, 126],
})
toy

In [ ]:
# 1) Find: look at all values incl. NaN – placeholders or simply wrong values stick out (here: 0)
print(toy["bpm"].value_counts(dropna=False).sort_index())
print(toy[(toy["bpm"] < 60) | (toy["bpm"] > 200)])        # range check

# 2) Decide & document: bpm 0 and empty artist mean "missing" → NaN
toy_clean = toy.copy()                                     # raw table stays untouched
toy_clean["bpm"] = toy_clean["bpm"].replace(0, np.nan)          # replace with "true" missing NaN
toy_clean["artist"] = toy_clean["artist"].replace("", np.nan)   # replace with "true" missing NaN
print("bpm missing:", toy["bpm"].isna().sum(), "→", toy_clean["bpm"].isna().sum())

In [ ]:
# Anything odd here?
charts["title"].value_counts().head(10)

In [ ]:
t = charts["title"]
checks = pd.DataFrame({
    "title":         t,
    "only_digits":   t.str.fullmatch(r"\d+", na=False),                 # e.g. "1981"
    "no_letters":    t.notna() & ~t.str.contains(r"[a-zA-Z]", na=False),
    "unusual_chars": t.str.contains(r"[#@*_~^|<>{}\\]", na=False),
    "stray_space":   t.notna() & (t.str.strip() != t),                  # NaN != NaN is True!
})
checks["any_odd"] = checks.drop(columns="title").any(axis=1)

print(checks.drop(columns="title").sum())                   # how many per check
odd = checks[checks["any_odd"]]
odd.to_csv(OUTPUT_DIR / "odd_titles.csv", index=False)      # collect for review
odd.head(20)

In [ ]:
# isna() → True/False per cell; True counts as 1
print(toy.isna().sum())    # absolute: missing values per column
print(toy.isna().mean())   # relative: share 0…1 (× 100 → %)

In [ ]:
# Placeholders are NOT NaN → check them explicitly
print((toy["bpm"] == 0).sum(), "× bpm == 0")
print((toy["artist"] == "").sum(), "× empty artist")

In [ ]:
# Per group: groupby
print(toy.groupby("year").size())               # rows per year
print(toy.groupby("year")["artist"].count())    # non-missing values per year (count ignores NaN)

In [ ]:
# Filter rows with a boolean mask
top100 = toy[toy["rank"] <= 100]

# Is a value in another collection? e.g. track IDs that have a file
available = {11, 44}
print(top100["track_id"].isin(available).tolist())

---
## 5 · Plotting with matplotlib
- `fig, ax = plt.subplots()` → a **figure** (the canvas) with one **axes** (the plot).
- Draw on `ax`: `ax.plot()` (lines), `ax.bar()` (bars), `ax.imshow()` (matrix as heatmap).
- Always label: `ax.set(xlabel=…, ylabel=…, title=…)`; shares → fix the range with `ylim=(0, 1)`.
- Save: `fig.savefig(OUTPUT_DIR / "name.png", dpi=150, bbox_inches="tight")`.

Toy numbers again – plot your own results the same way.

In [ ]:
years: list[int] = [2013, 2014, 2015]
share_a = [0.95, 0.80, 0.90]
share_b = [0.60, 0.70, 0.40]

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(years, share_a, marker="o", label="variable A")
ax.plot(years, share_b, marker="o", label="variable B")
ax.set(xlabel="Year", ylabel="Share", ylim=(0, 1), xticks=years, title="Line plot")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(years, [95, 80, 90])
ax.set(xlabel="Year", ylabel="Count", xticks=years, title="Bar plot")
fig.savefig(OUTPUT_DIR / "toy_bar.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Table (rows × columns) as heatmap
from typing import Any

table: Any = pd.DataFrame({"A": share_a, "B": share_b}, index=years)

fig, ax = plt.subplots(figsize=(6, 3))
im = ax.imshow(table.T, vmin=0, vmax=1, aspect="auto")   # .T → variables as rows
ax.set(xticks=range(len(table.index)), xticklabels=table.index,
       yticks=range(len(table.columns)), yticklabels=table.columns, title="Heatmap")
fig.colorbar(im, ax=ax, label="Share")
plt.show()

---
## Before you hand in your homework
- [ ] **Restart Kernel & Run All** runs without errors
- [ ] Downloaded files unchanged; new files only in `previews/` and `output/`
- [ ] Figures labelled (axes, units, title) and saved to `output/`
- [ ] Only hand in necessary files in one folder! 